# Notebook 2: XGBoost AQI Forecasting

This notebook trains an XGBoost regressor for monthly AQI forecasting using month/year, encoded city, lag AQI features (t-1, t-2), and weather attributes.

In [1]:
import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBRegressor

In [2]:
PROJECT_ROOT = Path('..').resolve()
DATA_DIR = PROJECT_ROOT / 'data'
MODEL_DIR = PROJECT_ROOT / 'backend' / 'models'
ARTIFACT_DIR = PROJECT_ROOT / 'backend' / 'artifacts'
MODEL_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

def read_csv_robust(path):
    for enc in ('utf-8', 'utf-8-sig', 'cp1252', 'latin1'):
        try:
            return pd.read_csv(path, encoding=enc)
        except UnicodeDecodeError:
            continue
    return pd.read_csv(path, encoding_errors='replace')

aqi_df = pd.read_csv(DATA_DIR / 'aqi_data.csv').rename(columns={'S02': 'SO2'})
weather_df = pd.read_csv(DATA_DIR / 'weather.csv').rename(columns={'Wind Speed': 'wind_speed'})
date_df = pd.read_csv(DATA_DIR / 'date_dimension.csv')
cities_df = read_csv_robust(DATA_DIR / 'cities.csv')

aqi_df = aqi_df.rename(columns={'AQI': 'aqi'})
weather_df = weather_df.rename(columns={
    'Temperature': 'temperature',
    'Humidity': 'humidity',
    'Rainfall': 'rainfall',
    'Visibility': 'visibility'
})
cities_df = cities_df.rename(columns={'State': 'state'})

In [3]:
feature_df = (
    aqi_df[['city_id', 'date_key', 'aqi']]
    .merge(weather_df[['city_id', 'date_key', 'temperature', 'humidity', 'wind_speed', 'rainfall', 'visibility']],
           on=['city_id', 'date_key'], how='inner')
    .merge(date_df[['date_key', 'year', 'month_number']], on='date_key', how='left')
    .merge(cities_df[['city_id', 'city_name']], on='city_id', how='left')
)

feature_df = feature_df.sort_values(['city_id', 'year', 'month_number']).reset_index(drop=True)
feature_df['aqi_lag_1'] = feature_df.groupby('city_id')['aqi'].shift(1)
feature_df['aqi_lag_2'] = feature_df.groupby('city_id')['aqi'].shift(2)

city_encoder = LabelEncoder()
feature_df['city_name'] = feature_df['city_name'].fillna('Unknown')
feature_df['city_encoded'] = city_encoder.fit_transform(feature_df['city_name'])

feature_cols = [
    'month_number', 'year', 'city_encoded',
    'aqi_lag_1', 'aqi_lag_2',
    'temperature', 'humidity', 'wind_speed', 'rainfall', 'visibility'
]

for col in feature_cols + ['aqi']:
    feature_df[col] = pd.to_numeric(feature_df[col], errors='coerce')

model_df = feature_df.dropna(subset=feature_cols + ['aqi']).copy()
model_df.head()

,city_id,date_key,aqi,temperature,humidity,wind_speed,rainfall,visibility,year,month_number,city_name,aqi_lag_1,aqi_lag_2,city_encoded
2,1,05-04,30,9,60.3,19.0,80.8,10000,2005,4,Aberdeen,37.0,45.0,0
3,1,05-05,26,14,64.4,17.9,97.0,10000,2005,5,Aberdeen,30.0,37.0,0
4,1,05-06,30,17,67.5,16.9,91.6,10000,2005,6,Aberdeen,26.0,30.0,0
5,1,05-07,34,20,70.7,15.8,102.4,8000,2005,7,Aberdeen,30.0,26.0,0
6,1,05-08,37,19,72.7,14.8,97.0,10000,2005,8,Aberdeen,34.0,30.0,0


In [4]:
max_year = int(model_df['year'].max())
split_year = max_year - 1

train_df = model_df[model_df['year'] <= split_year].copy()
test_df = model_df[model_df['year'] > split_year].copy()

if test_df.empty:
    split_idx = int(len(model_df) * 0.8)
    train_df = model_df.iloc[:split_idx].copy()
    test_df = model_df.iloc[split_idx:].copy()

X_train = train_df[feature_cols]
y_train = train_df['aqi']
X_test = test_df[feature_cols]
y_test = test_df['aqi']

xgb_model = XGBRegressor(
    n_estimators=300,
    learning_rate=0.07,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    objective='reg:squarederror',
    random_state=42,
    n_jobs=12,
    early_stopping_rounds=40
)

xgb_model.fit(
    X_train,
    y_train,
    eval_set=[(X_test, y_test)],
    verbose=False,
)

best_iteration = getattr(xgb_model, 'best_iteration', None)
if best_iteration is not None and int(best_iteration) >= 0:
    test_pred = xgb_model.predict(X_test, iteration_range=(0, int(best_iteration) + 1))
else:
    test_pred = xgb_model.predict(X_test)

mae = float(mean_absolute_error(y_test, test_pred))
rmse = float(np.sqrt(mean_squared_error(y_test, test_pred)))
r2 = float(r2_score(y_test, test_pred))

print(f'MAE : {mae:.4f}')
print(f'RMSE: {rmse:.4f}')
print(f'R2  : {r2:.4f}')
print(f'Best iteration: {best_iteration}')

MAE : 2.5578
RMSE: 6.3392
R2  : 0.9398
Best iteration: 117


In [5]:
xgb_model_path = MODEL_DIR / 'xgboost.pkl'
encoder_path = MODEL_DIR / 'city_label_encoder.pkl'
metadata_path = ARTIFACT_DIR / 'xgboost_metadata.json'

tmp_xgb_model_path = xgb_model_path.with_suffix(xgb_model_path.suffix + '.tmp')
tmp_encoder_path = encoder_path.with_suffix(encoder_path.suffix + '.tmp')
tmp_metadata_path = metadata_path.with_suffix(metadata_path.suffix + '.tmp')

xgb_metadata = {
    'model': 'XGBRegressor',
    'target': 'aqi',
    'features': feature_cols,
    'split_year': split_year,
    'best_iteration': int(best_iteration) if best_iteration is not None else None,
    'params': {
        'n_estimators': 300,
        'learning_rate': 0.07,
        'max_depth': 6,
        'subsample': 0.8,
        'colsample_bytree': 0.8,
        'n_jobs': 12,
        'random_state': 42,
        'early_stopping_rounds': 40
    },
    'metrics': {
        'mae': round(mae, 6),
        'rmse': round(rmse, 6),
        'r2': round(r2, 6)
    }
}

try:
    joblib.dump(xgb_model, tmp_xgb_model_path)
    joblib.dump(city_encoder, tmp_encoder_path)
    with open(tmp_metadata_path, 'w', encoding='utf-8') as fp:
        json.dump(xgb_metadata, fp, indent=2)

    tmp_xgb_model_path.replace(xgb_model_path)
    tmp_encoder_path.replace(encoder_path)
    tmp_metadata_path.replace(metadata_path)
finally:
    for tmp_path in [tmp_xgb_model_path, tmp_encoder_path, tmp_metadata_path]:
        if tmp_path.exists():
            tmp_path.unlink()

xgb_metadata

{'model': 'XGBRegressor',
 'target': 'aqi',
 'features': ['month_number',
  'year',
  'city_encoded',
  'aqi_lag_1',
  'aqi_lag_2',
  'temperature',
  'humidity',
  'wind_speed',
  'rainfall',
  'visibility'],
 'split_year': 2024,
 'best_iteration': 117,
 'params': {'n_estimators': 300,
  'learning_rate': 0.07,
  'max_depth': 6,
  'subsample': 0.8,
  'colsample_bytree': 0.8,
  'n_jobs': 12,
  'random_state': 42,
  'early_stopping_rounds': 40},
 'metrics': {'mae': 2.557753, 'rmse': 6.339187, 'r2': 0.939809}}

In [6]:
weather_cols = ['temperature', 'humidity', 'wind_speed', 'rainfall', 'visibility']
city_month_weather = model_df.groupby(['city_id', 'month_number'])[weather_cols].mean()
city_latest = model_df.sort_values(['city_id', 'year', 'month_number']).groupby('city_id').tail(1)

forecast_rows = []
best_iteration = getattr(xgb_model, 'best_iteration', None)

for _, row in city_latest.iterrows():
    city_id = int(row['city_id'])
    city_name = row['city_name']
    city_encoded = int(row['city_encoded'])

    prev_aqi_1 = float(row['aqi'])
    prev_aqi_2 = float(row['aqi_lag_1']) if pd.notna(row['aqi_lag_1']) else prev_aqi_1
    cur_year = int(row['year'])
    cur_month = int(row['month_number'])

    fallback_weather = {
        col: float(row[col]) if pd.notna(row[col]) else 0.0
        for col in weather_cols
    }

    for _step in range(12):
        next_month = 1 if cur_month == 12 else cur_month + 1
        next_year = cur_year + 1 if cur_month == 12 else cur_year

        weather_key = (city_id, next_month)
        if weather_key in city_month_weather.index:
            weather_values = city_month_weather.loc[weather_key].to_dict()
        else:
            weather_values = fallback_weather.copy()

        clean_weather = {}
        for col in weather_cols:
            value = weather_values.get(col)
            clean_weather[col] = float(value) if pd.notna(value) else fallback_weather[col]

        input_row = {
            'month_number': next_month,
            'year': next_year,
            'city_encoded': city_encoded,
            'aqi_lag_1': prev_aqi_1,
            'aqi_lag_2': prev_aqi_2,
            'temperature': clean_weather['temperature'],
            'humidity': clean_weather['humidity'],
            'wind_speed': clean_weather['wind_speed'],
            'rainfall': clean_weather['rainfall'],
            'visibility': clean_weather['visibility']
        }

        model_input = pd.DataFrame([input_row])[feature_cols]
        model_input = model_input.apply(pd.to_numeric, errors='coerce').replace([np.inf, -np.inf], np.nan)
        if model_input.isna().any(axis=None):
            continue

        if best_iteration is not None and int(best_iteration) >= 0:
            pred_raw = xgb_model.predict(model_input, iteration_range=(0, int(best_iteration) + 1))
        else:
            pred_raw = xgb_model.predict(model_input)

        pred_aqi = float(pred_raw[0])
        pred_aqi = max(0.0, min(500.0, pred_aqi))

        forecast_rows.append({
            'city_id': city_id,
            'city_name': city_name,
            'year': next_year,
            'month_number': next_month,
            'predicted_aqi': round(pred_aqi, 3)
        })

        prev_aqi_2 = prev_aqi_1
        prev_aqi_1 = pred_aqi
        cur_year = next_year
        cur_month = next_month

forecast_df = pd.DataFrame(forecast_rows)

forecast_path = ARTIFACT_DIR / 'future_aqi_predictions.csv'
tmp_forecast_path = forecast_path.with_suffix(forecast_path.suffix + '.tmp')

try:
    forecast_df.to_csv(tmp_forecast_path, index=False)
    tmp_forecast_path.replace(forecast_path)
finally:
    if tmp_forecast_path.exists():
        tmp_forecast_path.unlink()

forecast_df.head()

,city_id,city_name,year,month_number,predicted_aqi
0,1,Aberdeen,2026,1,29.739
1,1,Aberdeen,2026,2,27.335
2,1,Aberdeen,2026,3,22.859
3,1,Aberdeen,2026,4,19.092
4,1,Aberdeen,2026,5,16.586
